# VÜK source adapter

Control panel for `vuek.run` (`1_scrapers/vuek/src/run.py`). All logic lives in the module; this notebook only calls it. Every path comes from `core.paths`. vuek-gmbh.de needs no key and no login; plain HTTP works.

**What VÜK is.** VÜK GmbH, Ostwestfalen-Lippe: a recognized Überwachungsorganisation that was not in the project brief's list. Its site is a static Jimdo page without a locator. It states **five own stations** (Lage, Bielefeld-Brackwede, Bad Oeynhausen, Minden, Kirchlengern), one contact page each (`/kontakt/pruefstelle-<ort>/`). All five are own stations: `site_type = pruefstelle`, `in_scope = true`, `source_org = VÜK`, `location_id = VUEK-<slug>`.

**Completeness rule (the run fails loudly otherwise).** The station links of the *menu* (desktop and mobile), the *footer list* "VÜK Standorte", the *body text of /ueber-uns/*, the *XML sitemap* and the *HTML sitemap* are the same set, and the number the site states ("fünf eigenen Standorten", meta description and body text) equals the size of that set. The set is also compared with the last passing full run (the first run: with the five stations of the brief): a new or vanished station fails the run until you accept the change on purpose (`accept_station_change=True`). Every station page must yield street + PLZ + city.

**Coordinates.** The pages carry none (the Google map embed has only an address query), so the addresses are geocoded with OpenStreetMap Nominatim (at most 1 request per second, identifying User-Agent with contact). `coord_source` is `page` or `nominatim`; `plz_distance_km` compares them with the GeoNames PLZ centroid. A geocoder problem is a review finding, not a completeness failure.

**Change detection.** Every station page gets a normalized content hash (e-mail obfuscation decoded, scripts and cookie texts removed). The run summary lists which stations changed since the last passing full run: address, phone, hours, e-mail, services, notice, or "other page content".

**Known gap.** VÜK also inspects in partner workshops (PSPs). They are not listed anywhere on the site and out of scope; the run summary repeats the site's own sentence about them.

**Politeness.** One request at a time, 5-7 s apart (the site's robots.txt says `Crawl-Delay: 5`, honoured although robots.txt need not be obeyed), at most 3 retries with backoff, any Cloudflare page / 403 / 503 stops the run. No disallowed path (`/app/`, `/j/`) is requested.

| Mode | Requests | Goes to |
|---|---|---|
| `simulated` | none (fake site with 5 stations; negative tests below) | `test_output/` |
| `sample` | live, 2 to vuek-gmbh.de (`/ueber-uns/` + the first station page) + 1 to Nominatim; reports SAMPLE | `test_output/` |
| `full` | live, 11 to vuek-gmbh.de + 5 to Nominatim, about 2 minutes; needs `confirm_full_run=True` | `output/` |
| `rebuild` | none, re-parses the raw files of an earlier run folder | `output/` |

A run writes only `vuek_pruefstellen_<date>.csv` (the in-scope rows; `unverified` in the name if the run did not pass), `run_summary.json`/`.md`, `query_log.csv` and `raw/` (raw HTML per page, Nominatim answers). The files contain contact data and stay out of git. Coordinates: © OpenStreetMap contributors (ODbL).

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import contextlib
import io
import tempfile
from pathlib import Path

import pandas as pd

from core import paths
from core.geo import load_plz_centroids
from vuek import run as vuek
from vuek.simulate import SimulatedGeocoder, SimulatedVuek

ROOT = paths.project_root()
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)
CENTROIDS = load_plz_centroids(paths.geonames_dir() / "DE.txt")
ROOT

## 2. Offline: simulated run (no requests)

A fake vuek-gmbh.de with the real markup and synthetic data: 5 stations, the Cloudflare e-mail obfuscation re-keyed on every request. Must PASS with 5 stations. Writes one `simulated` run folder in `1_scrapers/vuek/test_output/`.

In [ ]:
sim = vuek.run(mode="simulated", centroids=CENTROIDS)
print(sim.outcome, "-", len(sim.frame), "stations in scope")
print(sim.run_dir.relative_to(ROOT))
pd.DataFrame(sim.summary["checks"])

### 2a. Negative tests: each of these must FAIL

The simulated site with one defect each (run in a temporary folder, nothing is written to the project). Every row must say FAIL and name the check that caught it; the last column shows what the check printed.

In [ ]:
FAULTS = {
    "menu_six": "the menu shows 6 stations",
    "footer_missing": "the footer lacks one station",
    "body_missing": "the body text lacks one station link",
    "stated_sechs": "the site states 'sechs'",
    "stated_unreadable": "the stated number cannot be read",
    "no_footer_block": "the footer block is gone",
    "sitemap_xml_extra": "the XML sitemap lists a sixth station",
    "sitemap_page_extra": "the HTML sitemap lists a sixth station",
    "no_address": "one station page has no address",
    "station_404": "one station page answers 404",
}
rows = []
for fault, what in FAULTS.items():
    with contextlib.redirect_stdout(io.StringIO()):  # the run prints its progress; not needed here
        r = vuek.run(mode="simulated", root=tempfile.mkdtemp(), client=SimulatedVuek(fault), centroids=CENTROIDS)
    hard = [c for c in r.summary["checks"] if not c["ok"] and c["severity"] == "hard"]
    rows.append({"fault": what, "outcome": r.outcome.split(":")[0], "first failed check": hard[0]["check"] if hard else "-",
                 "detail": (hard[0]["detail"] if hard else "-")[:90]})
neg = pd.DataFrame(rows)
assert (neg["outcome"] == "FAIL").all(), "a faulty site must never pass"
neg

## 3. Sample run (live, 3 requests)

**Live requests.** `/ueber-uns/` and the first station page from vuek-gmbh.de, plus one Nominatim request. Confirms that the site still answers with plain HTTP and that the parser still reads the pages. It does not fetch the sitemaps, so it can never prove completeness and always reports `SAMPLE`; it still FAILS on a problem it can see (menu/footer/body disagree, unreadable stated number, no address).

In [ ]:
sample = vuek.run(mode="sample", root=ROOT, centroids=CENTROIDS)  # 3 live requests
sample.frame[["name", "street", "zip", "city", "phone", "opening_hours", "lat", "lng", "coord_source", "plz_distance_km"]]

## 4. Full run (live, 11 + 5 requests, about 2 minutes)

**Live requests.** Needs `confirm_full_run=True`. The committed default is `False`: with it the cell stops with an error on purpose before any request is sent.

Order: `robots.txt`, `/ueber-uns/`, the HTML sitemap, `sitemap.xml`, `/kontakt/`, `/hauptuntersuchung/`, the five station pages (each followed by its Nominatim request). **PASS** needs every hard check: the five station sets agree, the stated number matches, the set equals the last passing full run (or the five stations of the brief), every station page yields street + PLZ + city. Review findings (HU not named on a page, no coordinates, coordinates far from the PLZ, notice text on a page, a phone number shared by several stations) are listed but do not fail the run.

If VÜK really opens or closes a station, the run fails on "station set equals ...". Check the site by hand, then run once with `accept_station_change=True`; that run becomes the new baseline.

In [ ]:
# Set confirm_full_run=True to start the full run. Keep False in the committed notebook.
full = vuek.run(mode="full", confirm_full_run=False, root=ROOT)

In [ ]:
# After a full run: outcome, hard failures, review findings, warnings
print(full.outcome, "-", len(full.frame), "stations in scope")
print("warnings:", full.summary["warnings"] or "none", "| drop check:", full.summary["drop_check"])
pd.DataFrame(full.summary["checks"]).query("not ok")

## 5. Inspect a run

Works on the last run of this kernel (`full` if it ran, else `sample`, else the simulated one).

In [ ]:
res = next((r for name in ("full", "sample", "sim") if (r := globals().get(name)) is not None and hasattr(r, "frame")), sim)
df, s = res.frame, res.summary
print(res.mode, res.outcome, "|", res.run_dir.relative_to(ROOT))
print("Counts:", s["counts"])
df[["location_id", "name", "street", "zip", "city", "state", "phone", "email", "coord_source", "plz_distance_km", "needs_review", "doubt_flags"]]

In [ ]:
# Opening hours: normalized and as published (raw text is always kept)
df[["source_id", "opening_hours", "opening_hours_raw", "notice"]]

In [ ]:
# Completeness: the station sets of every source
d = s["discovery"]
pd.DataFrame({
    "source": ["menu", "footer", "body text", "sitemap.xml", "HTML sitemap", "/kontakt/ page"],
    "stations": [len(d["menu"]), len(d["footer"]), len(d["body"]),
                 *[None if d[k] is None else len(d[k]) for k in ("sitemap_xml_station_urls", "sitemap_page_station_urls", "kontakt_hub")]],
})

In [ ]:
# Changes since the last passing full run (address, phone, hours, ...)
s["changes"]

In [ ]:
# Known gap, in the site's own words
print(s["known_gaps"][0])
print()
print("The site says:", d["partner_workshop_text"] or "(no sentence found)")

## 6. Offline rebuild from the raw files of a run

Re-parses and re-checks the raw HTML (and the saved Nominatim answers) of an earlier run folder: no request. Writes a `rebuild` run folder in `1_scrapers/vuek/output/`. Set `SOURCE` to a run folder name (e.g. the live `full` run or the `sample`); each execution makes a new folder.

In [ ]:
out = paths.output_dir("vuek", "full")
print("full / rebuild runs:", sorted(p.name for p in out.glob("*") if p.is_dir()))
SOURCE = None  # e.g. "2026-10-06_1130_full"
if SOURCE:
    rb = vuek.run(mode="rebuild", source_runs=[out / SOURCE])
    print(rb.outcome, "-", len(rb.frame), "stations;", rb.summary["requests_made"], "requests")
    display(pd.DataFrame(rb.summary["checks"]).query("not ok"))

## 7. Compare two runs (change detection on the exports)

Which stations appeared, disappeared or changed between two runs of `output/`. Set `OLD` and `NEW` to run-folder names.

In [ ]:
out = paths.output_dir("vuek", "full")
runs = sorted(p.name for p in out.glob("*") if (p / "run_summary.json").exists())
print("Runs in output/:", runs)

OLD, NEW = None, None  # e.g. "2026-10-06_1130_full", "2026-11-03_0900_full"
if OLD and NEW:
    def load(name):
        f = next((out / name).glob("vuek_pruefstellen_*.csv"))
        return pd.read_csv(f, sep=";", encoding="utf-8-sig", dtype=str, keep_default_na=False).set_index("source_id")
    old, new = load(OLD), load(NEW)
    print(f"{len(old)} -> {len(new)} | new: {sorted(set(new.index) - set(old.index))} | gone: {sorted(set(old.index) - set(new.index))}")
    cols = ["street", "zip", "city", "phone", "fax", "email", "opening_hours_raw", "services", "notice"]
    diff = [(i, c, old.at[i, c], new.at[i, c]) for i in old.index.intersection(new.index) for c in cols if old.at[i, c] != new.at[i, c]]
    display(pd.DataFrame(diff, columns=["station", "field", "old", "new"]))